# Moduł 08. Python Unit Tests — `pytest-mock` — v2.0

Autor: Kamil Bartocha

## Rozkład jazdy

- 🔹 1. Fixture `mocker` - prostszy interfejs do patch
- 🔹 2. `mocker.spy` - szpiegowanie prawdziwych metod
- 🔹 3. Kiedy `pytest-mock`, a kiedy `unittest.mock`

## 1. 🔹 Fixture `mocker` - prostszy interfejs do patch

Plugin `pytest-mock` dostarcza fixture `mocker`, który jest
wrapperem wokół `unittest.mock`. Główne zalety:

- brak potrzeby dekoratorów `@patch` - patch przez fixture
- automatyczne czyszczenie po każdym teście
- zwięzła składnia: `mocker.patch(...)` zamiast
  `with patch(...) as m:`

Instalacja: `pip install pytest-mock`

Podstawowe metody fixture `mocker`:

| Metoda | Odpowiednik w `unittest.mock` |
|---|---|
| `mocker.patch('path.to.obj')` | `patch('path.to.obj')` |
| `mocker.patch.object(obj, 'method')` | `patch.object(obj, 'method')` |
| `mocker.patch.dict(d, values)` | `patch.dict(d, values)` |
| `mocker.MagicMock()` | `MagicMock()` |
| `mocker.spy(obj, 'method')` | brak bezpośredniego odpowiednika |

Patch stworzony przez `mocker` jest automatycznie cofany
po zakończeniu testu - nie trzeba wywoływać `stop()`.

In [ ]:
import sys, subprocess, tempfile, os

def _run(code, flags=None):
    flags = flags or ['-v', '--tb=short', '-p', 'no:cacheprovider']
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='_test.py', delete=False, dir='.'
    ) as tmp:
        tmp.write(code)
        fname = tmp.name
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', fname] + flags,
        capture_output=True, text=True
    )
    short = os.path.basename(fname)
    print(result.stdout.replace(fname, short)[-1500:])
    os.unlink(fname)

# Przykład: mocker.patch zamiast @patch dekoratora
_run('''
import os

def count_files(directory):
    return len(os.listdir(directory))

def test_count_files(mocker):
    mocker.patch("os.listdir", return_value=["a.py", "b.py", "c.py"])
    assert count_files("/any/dir") == 3

def test_count_empty(mocker):
    mocker.patch("os.listdir", return_value=[])
    assert count_files("/empty") == 0
''')

# Przykład: mocker.patch.object i mocker.patch.dict
_run('''
import os

class Mailer:
    def send(self, to, subject):
        raise RuntimeError("prawdziwy mailer")

def notify_user(mailer, email):
    mailer.send(email, "Welcome!")
    return True

def test_notify_user(mocker):
    mailer = Mailer()
    mock_send = mocker.patch.object(mailer, "send")
    result = notify_user(mailer, "alice@example.com")
    assert result is True
    mock_send.assert_called_once_with("alice@example.com", "Welcome!")

def test_env_config(mocker):
    mocker.patch.dict(os.environ, {"MODE": "test"})
    assert os.environ["MODE"] == "test"
''')

---

### 🐍 Ćwiczenia — fixture `mocker`

1. Zamień `@patch(...)` dekorator na `mocker.patch(...)` w istniejących
   testach
2. Użyj `mocker.patch.object` do podmiany metody instancji
8. Sprawdź że mock jest automatycznie czyszczony między testami

In [ ]:
# Ćwiczenie 1: @patch -> mocker.patch
# Poniżej mamy test z @patch dekoratorem.
# Przepisz go do wersji używającej mocker.patch (bez dekoratora).

_run('''
import requests

def get_status(url):
    response = requests.get(url)
    return response.status_code

# Wersja do przepisania (z @patch):
# from unittest.mock import patch, Mock
# @patch("requests.get")
# def test_get_status(mock_get):
#     mock_get.return_value.status_code = 200
#     assert get_status("http://example.com") == 200

# TODO: przepisz test używając mocker.patch
def test_get_status(mocker):
    mock_get = mocker.patch(...)
    mock_get.return_value.status_code = ...
    assert get_status("http://example.com") == 200
''')

In [ ]:
# Ćwiczenie 2: mocker.patch.object
# Klasa Cache.get(key) normalnie odpytuje Redis.
# Użyj mocker.patch.object, by podmienić metodę get na instancji.

_run('''
class Cache:
    def get(self, key):
        raise RuntimeError("prawdziwy Redis")

    def set(self, key, value):
        raise RuntimeError("prawdziwy Redis")

def load_user(cache, user_id):
    data = cache.get(f"user:{user_id}")
    return data

def test_load_user_from_cache(mocker):
    cache = Cache()
    mock_get = mocker.patch.object(cache, ..., return_value={"name": "Alice"})
    user = load_user(cache, 1)
    assert user["name"] == "Alice"
    mock_get.assert_called_once_with(...)
''')

In [ ]:
# Ćwiczenie 8: automatyczne czyszczenie między testami
# Sprawdź że patch stworzony przez mocker NIE wycieka między testami.
# test_first patchuje os.getcwd(), test_second musi widzieć prawdziwą wartość.

_run('''
import os

def test_first(mocker):
    mocker.patch("os.getcwd", return_value="/fake/path")
    assert os.getcwd() == "/fake/path"

def test_second():
    # mocker patch z test_first powinien być już cofnięty
    real_cwd = os.getcwd()
    assert real_cwd != "/fake/path"
    print("real cwd:", real_cwd)
''')

## 2. 🔹 `mocker.spy` - szpiegowanie prawdziwych metod

`mocker.spy(obj, 'method')` to wzorzec spy wbudowany w `pytest-mock`.
Opakowuje prawdziwą metodę - wywołanie nadal wykonuje oryginalny kod,
ale mock rejestruje każde wywołanie i jego argumenty.

Różnica między spy a patch:

| | `mocker.patch` | `mocker.spy` |
|---|---|---|
| Wykonuje oryginalny kod | nie | **tak** |
| Rejestruje wywołania | tak | tak |
| Można zmienić return_value | tak | nie |

`mocker.stub()` tworzy pusty mock (podobny do `MagicMock`) bez spec,
przydatny gdy potrzebujemy prostego callbacku:

```python
callback = mocker.stub(name='on_success')
service.run(on_success=callback)
callback.assert_called_once()
```

In [ ]:
# Przykład: mocker.spy - prawdziwa metoda + rejestracja wywołań
_run('''
class MathService:
    def square(self, x):
        return x * x

    def sum_of_squares(self, numbers):
        return sum(self.square(n) for n in numbers)

def test_sum_of_squares_calls_square(mocker):
    svc = MathService()
    spy = mocker.spy(svc, "square")

    result = svc.sum_of_squares([1, 2, 3])

    # prawdziwy wynik (1 + 4 + 9 = 14)
    assert result == 14
    # spy zarejestrował 3 wywołania
    assert spy.call_count == 3
    print("call_count:", spy.call_count)
    print("call_args_list:", spy.call_args_list)
''')

# Przykład: mocker.stub jako callback
_run('''
class EventBus:
    def __init__(self):
        self._handlers = []

    def subscribe(self, handler):
        self._handlers.append(handler)

    def publish(self, event):
        for h in self._handlers:
            h(event)

def test_handler_called_on_publish(mocker):
    bus = EventBus()
    handler = mocker.stub(name="event_handler")
    bus.subscribe(handler)
    bus.publish({"type": "user_created", "id": 1})
    handler.assert_called_once_with({"type": "user_created", "id": 1})
    print("handler called:", handler.called)
''')

---

### 🐍 Ćwiczenia — mocker.spy i mocker.stub

4. Użyj `mocker.spy(obj, 'method')` i sprawdź `call_count` prawdziwej
   metody
7. Użyj `mocker.stub()` jako uproszczonego mocka bez spec
3. *(Trudniejsze)* Przetestuj `NotificationService` mockując trzy
   zależności przez `mocker`

In [ ]:
# Ćwiczenie 4: mocker.spy - call_count prawdziwej metody
# TextProcessor.process(text) wywołuje wewnętrznie self._clean(text)
# i self._tokenize(text). Użyj mocker.spy, by sprawdzić call_count
# bez zamiany prawdziwej logiki.

_run('''
class TextProcessor:
    def _clean(self, text):
        return text.strip().lower()

    def _tokenize(self, text):
        return text.split()

    def process(self, text):
        cleaned = self._clean(text)
        tokens = self._tokenize(cleaned)
        return tokens

def test_process_uses_clean_and_tokenize(mocker):
    proc = TextProcessor()
    spy_clean = mocker.spy(proc, ...)
    spy_tokenize = mocker.spy(proc, ...)

    result = proc.process("  Hello World  ")

    assert result == ["hello", "world"]
    assert spy_clean.call_count == ...
    assert spy_tokenize.call_count == ...
    print("tokens:", result)
''')

In [ ]:
# Ćwiczenie 7: mocker.stub()
# System zdarzeń przyjmuje callback on_ready(data).
# Użyj mocker.stub() jako callbacku i sprawdź że został wywołany
# z oczekiwanymi argumentami.

_run('''
class DataLoader:
    def load(self, source, on_ready):
        data = {"source": source, "rows": 100}
        on_ready(data)

def test_on_ready_callback(mocker):
    loader = DataLoader()
    on_ready = mocker.stub(name=...)

    loader.load("db://users", on_ready)

    on_ready.assert_called_once_with(...)
    print("stub called:", on_ready.called)
    print("call_args:", on_ready.call_args)
''')

In [ ]:
# Ćwiczenie 3: NotificationService - trzy zależności *(Trudniejsze)*
# NotificationService.notify(user_id, message) wykonuje:
#   1. user = user_repo.find(user_id)
#   2. email_client.send(user['email'], message)
#   3. audit_log.record(user_id, 'notification_sent')
# Przetestuj przez mocker, mockując wszystkie trzy zależności.

_run('''
class NotificationService:
    def __init__(self, user_repo, email_client, audit_log):
        self.user_repo = user_repo
        self.email_client = email_client
        self.audit_log = audit_log

    def notify(self, user_id, message):
        user = self.user_repo.find(user_id)
        self.email_client.send(user["email"], message)
        self.audit_log.record(user_id, "notification_sent")
        return True

def test_notify(mocker):
    user_repo = mocker.MagicMock()
    email_client = mocker.MagicMock()
    audit_log = mocker.MagicMock()

    user_repo.find.return_value = ...

    svc = NotificationService(user_repo, email_client, audit_log)
    result = svc.notify(1, "Hello!")

    assert result is True
    email_client.send.assert_called_once_with(...)
    audit_log.record.assert_called_once_with(1, "notification_sent")
    print("notify OK")
''')

## 3. 🔹 Kiedy `pytest-mock`, a kiedy `unittest.mock`

Oba narzędzia rozwiązują ten sam problem. Wybór zależy od kontekstu:

| Sytuacja | Wybór |
|---|---|
| Testy pytest, nowy projekt | `pytest-mock` (mocker) |
| Testy unittest.TestCase | `unittest.mock` (@patch) |
| Brak zewnętrznych zależności | `unittest.mock` |
| Spy na prawdziwych metodach | `mocker.spy` |
| Patch w fixture conftest.py | `mocker` (automatyczne czyszczenie) |

Wzorzec fixture wielokrotnego użytku (`conftest.py`):

```python
# conftest.py
@pytest.fixture
def mock_email(mocker):
    return mocker.patch('myapp.email.send')

# test_*.py
def test_signup_sends_email(mock_email):
    signup('alice@example.com')
    mock_email.assert_called_once()
```

Porównanie składni:

```python
# unittest.mock
@patch('myapp.requests.get')
def test_api(mock_get, ...):
    mock_get.return_value.json.return_value = {}

# pytest-mock
def test_api(mocker):
    mock_get = mocker.patch('myapp.requests.get')
    mock_get.return_value.json.return_value = {}
```

In [ ]:
# Przykład: ten sam test - @patch vs mocker.patch
_run('''
import requests
from unittest.mock import patch, Mock

def fetch_price(product_id):
    r = requests.get(f"https://api.shop.example.com/products/{product_id}")
    return r.json()["price"]

# Wersja 1: unittest.mock @patch
@patch("requests.get")
def test_fetch_price_with_decorator(mock_get):
    mock_get.return_value.json.return_value = {"price": 29.99}
    price = fetch_price(42)
    assert price == 29.99
    print("@patch version: OK")

# Wersja 2: pytest-mock mocker
def test_fetch_price_with_mocker(mocker):
    mock_get = mocker.patch("requests.get")
    mock_get.return_value.json.return_value = {"price": 29.99}
    price = fetch_price(42)
    assert price == 29.99
    print("mocker version: OK")
''')

# Przykład: fixture wielokrotnego użytku w conftest.py
_run('''
import pytest

class EmailService:
    def send(self, to, subject, body):
        raise RuntimeError("prawdziwy email")

email_service = EmailService()

def signup(email):
    email_service.send(email, "Welcome!", "Thanks for signing up.")
    return True

@pytest.fixture
def mock_email(mocker):
    return mocker.patch.object(email_service, "send")

def test_signup_sends_welcome(mock_email):
    signup("alice@example.com")
    mock_email.assert_called_once_with(
        "alice@example.com", "Welcome!", "Thanks for signing up."
    )
    print("fixture mock: OK")

def test_signup_returns_true(mock_email):
    result = signup("bob@example.com")
    assert result is True
    print("signup returns True: OK")
''')

---

### 🐍 Ćwiczenia — porównanie i wzorce

5. Porównaj: ten sam test przez dekorator `@patch` vs `mocker.patch`
6. *(Trudniejsze)* Fixture wielokrotnego użytku opakowująca `mocker.patch`
9. *(Trudniejsze)* Przetestuj `PaymentService` z mockowanym `BankClient`
   i `AuditLogger`

In [ ]:
# Ćwiczenie 5: @patch vs mocker.patch - porównanie
# Napisz dwie wersje tego samego testu:
#   - test_version_decorator: użyj @patch
#   - test_version_mocker: użyj mocker.patch
# Obie wersje testują funkcję get_config(key).

_run('''
import os
from unittest.mock import patch

def get_config(key):
    return os.environ.get(key, "default")

@patch("os.environ.get", return_value="test_value")
def test_version_decorator(mock_get):
    assert get_config("API_KEY") == ...
    print("@patch: OK")

def test_version_mocker(mocker):
    mocker.patch("os.environ.get", return_value=...)
    assert get_config("API_KEY") == "test_value"
    print("mocker: OK")
''')

In [ ]:
# Ćwiczenie 6: fixture wielokrotnego użytku *(Trudniejsze)*
# Utwórz fixture mock_storage opakowujący mocker.patch dla Storage.save.
# Użyj tej fixture w dwóch testach - bez powielania patch.

_run('''
import pytest

class Storage:
    def save(self, key, data):
        raise RuntimeError("prawdziwy zapis")

    def load(self, key):
        raise RuntimeError("prawdziwy odczyt")

storage = Storage()

def save_user(user):
    storage.save(f"user:{user["id"]}", user)
    return True

@pytest.fixture
def mock_storage(mocker):
    return mocker.patch.object(storage, ...)

def test_save_user_calls_storage(mock_storage):
    save_user({"id": 1, "name": "Alice"})
    mock_storage.assert_called_once_with("user:1", ...)
    print("fixture test 1: OK")

def test_save_user_returns_true(mock_storage):
    result = save_user({"id": 2, "name": "Bob"})
    assert result is True
    print("fixture test 2: OK")
''')

In [ ]:
# Ćwiczenie 9: PaymentService z BankClient i AuditLogger *(Trudniejsze)*
# PaymentService.charge(user_id, amount) wykonuje:
#   1. result = bank_client.process_payment(user_id, amount)
#   2. jeśli result.success: audit_logger.log('payment_ok', user_id, amount)
#   3. jeśli nie: audit_logger.log('payment_failed', user_id, amount)
#   4. zwraca result.success
# Napisz dwa testy: sukces i niepowodzenie, używając mocker.

_run('''
class PaymentService:
    def __init__(self, bank_client, audit_logger):
        self.bank_client = bank_client
        self.audit_logger = audit_logger

    def charge(self, user_id, amount):
        result = self.bank_client.process_payment(user_id, amount)
        if result.success:
            self.audit_logger.log("payment_ok", user_id, amount)
        else:
            self.audit_logger.log("payment_failed", user_id, amount)
        return result.success

def test_charge_success(mocker):
    bank = mocker.MagicMock()
    audit = mocker.MagicMock()
    bank.process_payment.return_value.success = True

    svc = PaymentService(bank, audit)
    result = svc.charge(1, 99.99)

    assert result is True
    audit.log.assert_called_once_with(...)
    print("charge_success: OK")

def test_charge_failure(mocker):
    bank = mocker.MagicMock()
    audit = mocker.MagicMock()
    bank.process_payment.return_value.success = False

    svc = PaymentService(bank, audit)
    result = svc.charge(2, 49.99)

    assert result is False
    audit.log.assert_called_once_with(...)
    print("charge_failure: OK")
''')